In [11]:
"""
Implementation of XGBOOST for IPO underpricing prediciton.

Implementation based on: 
- https://xgboost.readthedocs.io/en/release_3.2.0/
"""

import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")



Device: cuda


## Load Data


In [12]:
full = pd.read_csv('ipologists/data/final/dataset_full.csv')
risk = pd.read_csv('ipologists/data/final/dataset_with_risk.csv')
print(f"Full Dataset: {full.shape}")
print(f"Risk Dataset: {risk.shape}")

Full Dataset: (6110, 24)
Risk Dataset: (862, 28)


In [13]:
full.head()

,Pricing Date,Issuer Name,Offer Size (M),Offer Price,Offer To 1st Close,Initial Pub Offer (Shares Offered),Industry Sector,Market Cap at Offer (M),Instit Owner (% Shares Out),Primary Exchange,...,has_bulge_bracket,vix,nasdaq,fed_funds,treasury_10y,cpi,unemployment,gdp,ipo_volume,market_return_1m
0,2000-01-24,Neoforma Inc,91.00,13.0,302.884613,7000000.0,Technology,732.744,0.014765,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
1,2000-01-24,Townsquare Media 2010 Inc,136.00,8.5,0.000000,16000000.0,Communications,272.983,NaN,,...,0,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
2,2000-01-25,Healthgate Data Corp,41.25,11.0,6.818182,3750000.0,Communications,180.900,NaN,,...,0,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
3,2000-01-25,T/R Systems Inc,30.00,10.0,59.380001,3000000.0,Technology,115.002,NaN,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
4,2000-01-26,John Hancock Financial Services Inc,1734.00,17.0,3.676471,102000000.0,Financial,5638.900,0.072897,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN


In [14]:

risk.head()

,Pricing Date,Issuer Name,Offer Size (M),Offer Price,Offer To 1st Close,Initial Pub Offer (Shares Offered),Industry Sector,Market Cap at Offer (M),Instit Owner (% Shares Out),Primary Exchange,...,treasury_10y,cpi,unemployment,gdp,ipo_volume,market_return_1m,regulatory_risk,competitive_risk,financial_risk,overall_risk
0,2000-01-27,Packaging Corp of America,555.000,12.0,0.000000,46250000.0,Industrial,1232.700,101.9270,New York,...,6.68,NaN,NaN,NaN,NaN,NaN,7,3,8,6
1,2000-02-03,Agenus Inc,63.000,18.0,240.972229,3500000.0,"Consumer, Non-cyclical",426.886,26.5410,NASDAQ CM,...,6.42,170.0,4.1,NaN,91.4042,NaN,6,5,9,7
2,2000-02-10,Beasley Broadcast Group Inc,106.175,15.5,-8.870968,6850000.0,Communications,0.000,24.3877,NASDAQ CM,...,6.42,170.0,4.1,NaN,91.4042,NaN,6,8,9,8
3,2000-04-06,Sangamo Therapeutics Inc,52.500,15.0,0.416667,3500000.0,"Consumer, Non-cyclical",334.500,16.3823,NASDAQ CM,...,6.23,171.0,4.0,NaN,92.9534,-0.026372,5,8,7,7
4,2000-04-06,LivePerson Inc,32.000,8.0,18.750000,4000000.0,Technology,234.720,32.8110,NASDAQ GS,...,6.23,171.0,4.0,NaN,92.9534,-0.026372,1,7,8,7


## Preprocessing


In [15]:
class DataPrepPipeline:
  def __init__(self,features):
    self.features = features
  def fit(self, X):
    return self       # XGBoost automatically deals with NA values so no need to do anything
  def transform(self, X):
    return X[self.features].values.astype(np.float32)

In [16]:

features = ['Offer Size (M)', 'Offer Price', 'Initial Pub Offer (Shares Offered)',
    'Market Cap at Offer (M)', 'offer_size_to_mktcap', 'has_bulge_bracket',
    'vix', 'nasdaq', 'fed_funds', 'treasury_10y', 'cpi',
    'unemployment', 'gdp', 'ipo_volume', 'market_return_1m'
]

risk_features = features + ['regulatory_risk', 'competitive_risk', 'financial_risk', 'overall_risk']

In [17]:
# Full Dataset
full_clean = full.dropna(subset=['underpriced'])
risk_clean = risk.dropna(subset=['underpriced'])
X_df_full = full_clean.drop(columns=[
    'underpriced', 'Offer To 1st Close',
    'Pricing Date', 'Issuer Name', 'ticker',
    'Primary Exchange', 'Instit Owner (% Shares Out)',
    'Industry Sector', 'lead_bookrunner'
])
y_df_full = full_clean['underpriced']

#This is the 80/20 from lecture notes 9
train_ix = X_df_full.sample(frac=0.8, random_state=42).index
test_ix = X_df_full.drop(train_ix).index

X_train_df_full = X_df_full.loc[train_ix]
y_train_df_full = y_df_full.loc[train_ix]

X_test_df_full  = X_df_full.loc[test_ix]
y_test_df_full  = y_df_full.loc[test_ix]

pipeline_full = DataPrepPipeline(X_train_df_full.columns.tolist())
pipeline_full.fit(X_train_df_full)

X_train_full = pipeline_full.transform(X_train_df_full)
X_test_full  = pipeline_full.transform(X_test_df_full)
y_train_full = y_train_df_full.values
y_test_full  = y_test_df_full.values

In [18]:
# Risk Dataset
risk_clean = risk.dropna(subset=['underpriced'])

X_df_risk = risk_clean.drop(columns=[
    'underpriced', 'Offer To 1st Close',
    'Pricing Date', 'Issuer Name', 'ticker',
    'Primary Exchange', 'Instit Owner (% Shares Out)',
    'Industry Sector', 'lead_bookrunner'
])
y_df_risk = risk_clean['underpriced']
#This is the 80/20 from lecture notes 9
train_ix_r = X_df_risk.sample(frac=0.8, random_state=42).index
test_ix_r = X_df_risk.drop(train_ix_r).index

X_train_df_risk = X_df_risk.loc[train_ix_r]
y_train_df_risk = y_df_risk.loc[train_ix_r]
X_test_df_risk  = X_df_risk.loc[test_ix_r]
y_test_df_risk  = y_df_risk.loc[test_ix_r]

pipeline_risk = DataPrepPipeline(X_train_df_risk.columns.tolist())
pipeline_risk.fit(X_train_df_risk)

X_train_risk = pipeline_risk.transform(X_train_df_risk)
X_test_risk  = pipeline_risk.transform(X_test_df_risk)
y_train_risk = y_train_df_risk.values
y_test_risk  = y_test_df_risk.values

## Binary Model Full Dataset


In [25]:
overpriced = (y_train_full ==0).sum()
underpriced = (y_train_full ==1).sum()

model_full = XGBClassifier(n_estimators=500, max_depth= 6, learning_rate = 0.05, scale_pos_weight=overpriced/underpriced )

model_full.fit(X_train_full, y_train_full, eval_set=[(X_test_full, y_test_full)], verbose=False)
preds_full = model_full.predict(X_test_full)
print(f"Accuracy: {accuracy_score(y_test_full, preds_full):.3f}")

Accuracy: 0.717


## Binary Model Risk Dataset

In [35]:
overpriced_risk = (y_train_risk ==0).sum()
underpriced_risk = (y_train_risk ==1).sum()

model_risk = XGBClassifier(n_estimators=200, max_depth= 5, learning_rate = 0.05, scale_pos_weight=overpriced_risk/underpriced_risk )

model_risk.fit(X_train_risk, y_train_risk, eval_set=[(X_test_risk, y_test_risk)], verbose=False)
preds_risk = model_risk.predict(X_test_risk)
print(f"Accuracy: {accuracy_score(y_test_risk, preds_risk):.3f}")


Accuracy: 0.698


## 3-Class Model Full Dataset